In [134]:
# https://www.kaggle.com/datasets/vanshtyagi2o1/dataset-diseases-and-symptoms-trainingtesting?select=Training.csv
import tkinter as tk
import customtkinter as ctk
import pandas as pd
import numpy as np
import spacy
from thefuzz import process, fuzz
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import threading
import random
import re
from spellchecker import SpellChecker

In [135]:
# Load NLP globally for performance
try:
    nlp = spacy.load("en_core_web_md")
except:
    import os
    os.system("python -m spacy download en_core_web_md")
    nlp = spacy.load("en_core_web_md")

In [ ]:
# IMPORTANT: Always sort by length after defining
MEDICAL_MAP = {

    # ================= SKIN =================
    "severe itching": "itching",
    "itching": "itching", "itchy": "itching",
    "internal itching": "internal_itching",

    "skin rash": "skin_rash", "rash": "skin_rash",
    "nodules": "nodal_skin_eruptions", "skin bumps": "nodal_skin_eruptions",

    "dark patches": "dischromic_patches",
    "skin discoloration": "dischromic_patches",

    "pus pimples": "pus_filled_pimples", "pimples": "pus_filled_pimples",
    "blackheads": "blackheads",
    "skin peeling": "skin_peeling", "flaky skin": "skin_peeling",

    "silver flakes": "silver_like_dusting",
    "nail dents": "small_dents_in_nails",
    "inflamed nails": "inflammatory_nails",

    "blisters": "blister",
    "nose sores": "red_sore_around_nose",
    "yellow crust": "yellow_crust_ooze",
    "red body spots": "red_spots_over_body",

    # ================= FEVER / GENERAL =================
    "high fever": "high_fever",
    "fever": "high_fever", "burning up": "high_fever",
    "mild fever": "mild_fever",

    "chills": "chills", "cold chills": "chills",
    "shivering": "shivering",

    "very tired": "fatigue",
    "fatigue": "fatigue", "tired": "fatigue",
    "lethargic": "lethargy",

    "weak": "malaise", "weakness": "malaise",

    "weight gain": "weight_gain",
    "losing weight": "weight_loss",

    "excess hunger": "increased_appetite",
    "always hungry": "increased_appetite",

    "dehydrated": "dehydration", "very thirsty": "dehydration",

    "sweating a lot": "sweating",

    # ================= RESPIRATORY =================
    "continuous sneezing": "continuous_sneezing",
    "sneezing": "continuous_sneezing",

    "dry cough": "cough", "coughing": "cough",

    "phlegm": "phlegm",
    "thick sputum": "mucoid_sputum",

    "rust colored sputum": "rusty_sputum",
    "blood cough": "blood_in_sputum",

    "shortness of breath": "breathlessness",
    "breathless": "breathlessness",

    "runny nose": "runny_nose",
    "blocked nose": "congestion",

    "sinus pain": "sinus_pressure",
    "sore throat": "throat_irritation",
    "throat pain": "throat_irritation",
    "throat patches": "patches_in_throat",

    # ================= DIGESTIVE =================
    "stomach pain": "stomach_pain",
    "belly pain": "belly_pain",
    "abdomen pain": "abdominal_pain",

    "acid reflux": "acidity", "heartburn": "acidity",

    "mouth ulcers": "ulcers_on_tongue",

    "vomiting": "vomiting",
    "feeling nauseous": "nausea",

    "indigestion": "indigestion",

    "no appetite": "loss_of_appetite",
    "not eating": "loss_of_appetite",

    "constipation": "constipation",
    "loose motion": "diarrhoea",

    "gas problem": "passage_of_gases",

    "bloody stool": "bloody_stool",
    "anal pain": "pain_in_anal_region",
    "anus irritation": "irritation_in_anus",

    "stomach bleeding": "stomach_bleeding",
    "bloated stomach": "distention_of_abdomen",

    # ================= NEURO =================
    "headache": "headache",
    "pain behind eyes": "pain_behind_the_eyes",

    "dizzy": "dizziness",
    "spinning": "spinning_movements",

    "loss of balance": "loss_of_balance",
    "unsteady walking": "unsteadiness",

    "slurred speech": "slurred_speech",

    "confused": "altered_sensorium",

    "can't focus": "lack_of_concentration",

    "blur vision": "blurred_and_distorted_vision",
    "vision issues": "visual_disturbances",

    "loss of smell": "loss_of_smell",

    # ================= MUSCLE =================
    "joint pain": "joint_pain",
    "knee pain": "knee_pain",
    "hip pain": "hip_joint_pain",

    "muscle pain": "muscle_pain",
    "muscle weakness": "muscle_weakness",

    "stiff neck": "stiff_neck",
    "neck pain": "neck_pain",

    "back pain": "back_pain",

    "joint stiffness": "movement_stiffness",
    "swollen joints": "swelling_joints",

    "weak limbs": "weakness_in_limbs",

    # ================= URINARY =================
    "burning urination": "burning_micturition",
    "burning pee": "burning_micturition",

    "frequent urination": "continuous_feel_of_urine",
    "pee often": "continuous_feel_of_urine",

    "dark pee": "dark_urine",
    "yellow pee": "yellow_urine",

    "smelly urine": "foul_smell_of_urine",
    "bladder pain": "bladder_discomfort",

    # ================= CARDIO =================
    "chest pain": "chest_pain",

    "fast heartbeat": "fast_heart_rate",
    "heart racing": "fast_heart_rate",

    "palpitations": "palpitations",

    "swollen legs": "swollen_legs",
    "swollen veins": "swollen_blood_vessels",

    "bruising": "bruising",

    "puffy face": "puffy_face_and_eyes",

    # ================= LIVER =================
    "yellow skin": "yellowish_skin",
    "yellow eyes": "yellowing_of_eyes",

    "liver failure": "acute_liver_failure",

    # ================= OTHER =================
    "fluid retention": "fluid_overload",

    "family disease history": "family_history",

    "alcohol use": "history_of_alcohol_consumption",

    "blood transfusion": "receiving_blood_transfusion",

    "unsterile injection": "receiving_unsterile_injections"
}

# 🔥 CRITICAL STEP (DO NOT SKIP)
MEDICAL_MAP = dict(sorted(MEDICAL_MAP.items(), key=lambda x: len(x[0]), reverse=True))

In [137]:
class MedicalEngine:
    def __init__(self, csv_path):
        # 1. Load Data
        self.df = pd.read_csv(csv_path)
        self.df = self.df.loc[:, ~self.df.columns.str.contains('^Unnamed')]
        
        self.le = LabelEncoder()
        self.df['prognosis'] = self.le.fit_transform(self.df['prognosis'])
        
        self.X = self.df.drop('prognosis', axis=1)
        self.y = self.df['prognosis']
        
        # 2. Train Model
        self.model = RandomForestClassifier(n_estimators=250, random_state=42)
        self.model.fit(self.X, self.y)
        
        # 3. Setup NLP Tools
        self.symptoms_list = list(self.X.columns)
        self.clean_symptoms = [s.replace('_', ' ') for s in self.symptoms_list]
        self.spell = SpellChecker()

    def correct_spelling(self, text):
        """NLP FEATURE 1: Corrects misspelled words dynamically."""
        words = text.split()
        corrected_words = []
        for word in words:
            # Ignore very short words or known medical terms
            if len(word) > 3 and word not in MEDICAL_MAP:
                correction = self.spell.correction(word)
                corrected_words.append(correction if correction else word)
            else:
                corrected_words.append(word)
        return " ".join(corrected_words)
    
    def is_negated(self, doc, start_idx, end_idx):
        """
        Checks if a span of tokens in the doc is negated.
        start_idx and end_idx are the token offsets for the symptom phrase.
        """
        for i in range(start_idx, end_idx):
            token = doc[i]
            
            # 1. Check immediate children for negation dependency
            for child in token.children:
                if child.dep_ == "neg" or child.text.lower() in ["no", "not", "never", "dont", "doesnt"]:
                    return True
            
            # 2. Check the head (parent) of the token
            # If 'sore' is part of 'sore throat', and 'throat' is negated, 'sore' is negated.
            if token.head.text.lower() in ["no", "not", "never"]:
                return True
                
        return False

    def extract_symptoms(self, text):
        """NLP FEATURE 4: Multi-layered Accumulative Extraction (Fixed Negation)."""
        
        # 1. Pre-processing
        text_lower = text.lower().strip()
        # IMPORTANT: We do NOT strip punctuation here because SpaCy needs it 
        # to understand where sentences end (negation shouldn't cross a period).
        clean_text = self.correct_spelling(text_lower)
        
        # Initial SpaCy doc
        doc = nlp(clean_text)
        found_symptoms = []
        
        # We use a set of 'masked' token indices to prevent Layer 2/3 
        # from picking up words already caught in Layer 1.
        masked_indices = set()

        # --- Layer 1: Dictionary Match (Longest-First) ---
        sorted_keys = sorted(MEDICAL_MAP.keys(), key=len, reverse=True)

        for key in sorted_keys:
            # We look for the key in the doc tokens
            # This is more reliable than regex for handling negation
            key_tokens = key.split()
            key_len = len(key_tokens)
            
            for i in range(len(doc) - key_len + 1):
                window = doc[i : i + key_len]
                if window.text.lower() == key.lower():
                    # Check if these tokens are already masked
                    if any(idx in masked_indices for idx in range(i, i + key_len)):
                        continue
                    
                    # BUG FIX: Check negation using the dependency tree
                    if not self.is_negated(doc, i, i + key_len):
                        found_symptoms.append(MEDICAL_MAP[key])
                    
                    # Mask these tokens
                    for idx in range(i, i + key_len):
                        masked_indices.add(idx)

        # --- Layer 2: Semantic NLP Matching (Remainder Processing) ---
        # Create a 'remainder' string from unmasked, non-stopword tokens
        remaining_tokens = [t for i, t in enumerate(doc) if i not in masked_indices and not t.is_stop and len(t.text) > 2]
        
        if remaining_tokens:
            query_vec = nlp(" ".join([t.text for t in remaining_tokens]))
            
            best_match = None
            highest_similarity = 0.0

            for idx, sym in enumerate(self.clean_symptoms):
                sym_vec = nlp(sym)
                if sym_vec.has_vector and query_vec.has_vector:
                    similarity = query_vec.similarity(sym_vec)
                    if similarity > highest_similarity:
                        highest_similarity = similarity
                        best_match = self.symptoms_list[idx]

            if highest_similarity > 0.84: 
                found_symptoms.append(best_match)
            
            # --- Layer 3: Fuzzy Matching Fallback ---
            elif len(query_vec.text) > 3:
                match, score = process.extractOne(query_vec.text, self.clean_symptoms, scorer=fuzz.token_set_ratio)
                if score > 88:
                    found_symptoms.append(self.symptoms_list[self.clean_symptoms.index(match)])

        return list(set(found_symptoms))

    def get_recommendations(self, current_symptoms, top_n=3):
        """NLP FEATURE 2: Contextual Recommendation Engine."""
        if not current_symptoms: return []
        
        mask = np.ones(len(self.X), dtype=bool)
        for s in current_symptoms:
            if s in self.symptoms_list:
                mask = mask & (self.X[s] == 1)
        
        subset = self.X[mask]
        
        if len(subset) == 0:
            mask = np.zeros(len(self.X), dtype=bool)
            for s in current_symptoms:
                if s in self.symptoms_list:
                    mask = mask | (self.X[s] == 1)
            subset = self.X[mask]
            
        if len(subset) == 0: return []

        counts = subset.sum(axis=0).drop(labels=current_symptoms, errors='ignore')
        top_symptoms = counts.nlargest(top_n).index.tolist()
        
        return [s.replace('_', ' ') for s in top_symptoms if counts[s] > 0]

    def predict_top_two(self, symptoms):
        if not symptoms: return []
        vector = np.zeros(len(self.symptoms_list))
        for s in symptoms:
            if s in self.symptoms_list:
                vector[self.symptoms_list.index(s)] = 1
        
        prob = self.model.predict_proba([vector])[0]
        top2_idx = np.argsort(prob)[-2:][::-1] 
        
        results = []
        for idx in top2_idx:
            disease = self.le.inverse_transform([idx])[0]
            confidence = prob[idx] * 100
            results.append((disease, confidence))
            
        return results
        
    def run_system_test(self, test_file_path):
        print(f"\n--- INITIATING SYSTEM VALIDATION ({test_file_path}) ---")
        try:
            from sklearn.metrics import accuracy_score
            test_df = pd.read_csv(test_file_path)
            test_df = test_df.loc[:, ~test_df.columns.str.contains('^Unnamed')]
            
            X_test = test_df.drop('prognosis', axis=1)
            y_test_text = test_df['prognosis']
            
            y_test_numeric = self.le.transform(y_test_text)
            y_pred = self.model.predict(X_test)
            
            acc = accuracy_score(y_test_numeric, y_pred)
            print(f"STATUS: Validation Complete. Accuracy: {acc * 100:.2f}%")
        except Exception as e:
            print(f"TEST ERROR: {e}")

In [138]:
class MedicalUI(ctk.CTk):
    def __init__(self, engine):
        super().__init__()
        self.engine = engine
        self.user_symptoms = []

        self.title("AI Medical Assistant")
        self.geometry("650x850")
        ctk.set_appearance_mode("dark")

        # Layout
        self.grid_rowconfigure(0, weight=1)
        self.grid_columnconfigure(0, weight=1)

        # Chat Window
        self.chat_display = ctk.CTkTextbox(self, state="disabled", font=("Segoe UI", 14), wrap="word")
        self.chat_display.grid(row=0, column=0, padx=20, pady=20, sticky="nsew")

        # Input Area
        self.input_frame = ctk.CTkFrame(self)
        self.input_frame.grid(row=1, column=0, padx=20, pady=(0, 10), sticky="ew")

        self.entry = ctk.CTkEntry(self.input_frame, placeholder_text="Enter symptoms here...", height=45)
        self.entry.pack(side="left", fill="x", expand=True, padx=(0, 10))
        self.entry.bind("<Return>", lambda e: self.send_action())

        self.send_btn = ctk.CTkButton(self.input_frame, text="Send", command=self.send_action, width=90, height=45)
        self.send_btn.pack(side="right")

        # Bottom Action Panel
        self.action_frame = ctk.CTkFrame(self, fg_color="transparent")
        self.action_frame.grid(row=2, column=0, padx=20, pady=(0, 20), sticky="ew")
        
        self.manage_btn = ctk.CTkButton(self.action_frame, text="Manage Symptoms", fg_color="#3a3a3a", hover_color="#555555", command=self.open_symptom_manager)
        self.manage_btn.pack(side="left", expand=True, fill="x", padx=(0, 10))

        self.diag_btn = ctk.CTkButton(self.action_frame, text="GENERATE DIAGNOSIS", fg_color="#1a5e1a", hover_color="#124012", command=self.generate_diagnosis_report)
        self.diag_btn.pack(side="right", expand=True, fill="x")

        self.add_log("System", "Online. Describe your symptoms (e.g., 'I have a headache and sore throat').")

    def add_log(self, sender, text):
        self.chat_display.configure(state="normal")
        self.chat_display.insert("end", f"{sender}: {text}\n\n")
        self.chat_display.configure(state="disabled")
        self.chat_display.see("end")

    def send_action(self):
        msg = self.entry.get()
        if not msg: return
        self.add_log("You", msg)
        self.entry.delete(0, "end")
        threading.Thread(target=self.process_input, args=(msg,)).start()

    def process_input(self, text):
        clean_text = text.lower().strip()

        if any(x in clean_text for x in ["diagnose", "done", "finish"]):
            self.after(0, self.generate_diagnosis_report)
            return

        extracted = self.engine.extract_symptoms(clean_text)
        
        if extracted:
            new_symptoms = [s for s in extracted if s not in self.user_symptoms]
            self.user_symptoms.extend(new_symptoms)
            
            if new_symptoms:
                names = ", ".join([n.replace('_', ' ') for n in new_symptoms])
                res = f"Recorded: {names}."
                
                # Fetch dynamically recommended symptoms
                recs = self.engine.get_recommendations(self.user_symptoms)
                if recs:
                    res += f"\n\nPatients with these symptoms often also report:\n• " + "\n• ".join(recs) + "\n\nAre you experiencing any of these?"
            else:
                res = "I already have that on record. Any other symptoms?"
        else:
            if any(g in clean_text for g in ["hi", "hello", "hey"]):
                res = "Hello! Please describe your symptoms so I can help."
            else:
                res = "I didn't recognize a medical symptom. Could you rephrase?"

        self.after(0, lambda: self.add_log("Assistant", res))

    # --- NEW FEATURE: SYMPTOM MANAGER ---
    def open_symptom_manager(self):
        manager = ctk.CTkToplevel(self)
        manager.title("Active Symptoms")
        manager.geometry("350x400")
        manager.grab_set() # Focus lock
        
        ctk.CTkLabel(manager, text="Current Recorded Symptoms", font=("Segoe UI", 16, "bold")).pack(pady=10)
        
        scroll_frame = ctk.CTkScrollableFrame(manager)
        scroll_frame.pack(fill="both", expand=True, padx=10, pady=10)

        if not self.user_symptoms:
            ctk.CTkLabel(scroll_frame, text="No symptoms recorded yet.").pack(pady=20)
            return

        def remove_symptom(sym, frame):
            self.user_symptoms.remove(sym)
            frame.destroy()
            self.add_log("System", f"Removed '{sym.replace('_', ' ')}' from records.")

        for sym in self.user_symptoms:
            row = ctk.CTkFrame(scroll_frame, fg_color="#2b2b2b")
            row.pack(fill="x", pady=2)
            
            ctk.CTkLabel(row, text=sym.replace('_', ' ').title(), font=("Segoe UI", 13)).pack(side="left", padx=10, pady=8)
            del_btn = ctk.CTkButton(row, text="❌", width=30, fg_color="#801919", hover_color="#5c1111", 
                                    command=lambda s=sym, f=row: remove_symptom(s, f))
            del_btn.pack(side="right", padx=10)

   # --- NEW FEATURE: MODERN DIAGNOSIS GRID (Updated with Symptoms) ---
    def generate_diagnosis_report(self):
        if not self.user_symptoms:
            self.add_log("Assistant", "No symptoms recorded. Please describe how you feel first.")
            return
            
        # 1. Capture data before clearing
        current_symptoms = list(self.user_symptoms) 
        symptom_text = ", ".join(current_symptoms).replace("_", " ").title()
        predictions = self.engine.predict_top_two(current_symptoms)
        
        # 2. Window Setup
        report_win = ctk.CTkToplevel(self)
        report_win.title("Preliminary Assessment Report")
        report_win.geometry("700x550") # Increased height slightly for the symptom list
        report_win.grab_set()

        # Header
        ctk.CTkLabel(report_win, text="DIAGNOSIS REPORT", font=("Segoe UI", 24, "bold")).pack(pady=(20, 5))
        ctk.CTkLabel(report_win, text="Based on the symptom data provided.", text_color="gray").pack(pady=(0, 10))

        # --- NEW: Symptom List Section ---
        symptom_box = ctk.CTkFrame(report_win, fg_color="#161b22", corner_radius=10)
        symptom_box.pack(fill="x", padx=30, pady=10)
        
        ctk.CTkLabel(symptom_box, text="SYMPTOMS ANALYZED", font=("Segoe UI", 10, "bold"), text_color="#888888").pack(pady=(5, 0))
        # Use wraplength to ensure long lists of symptoms don't break the UI
        ctk.CTkLabel(symptom_box, text=symptom_text, font=("Segoe UI", 13, "italic"), 
                     text_color="#cbd5e0", wraplength=600).pack(pady=(5, 10), padx=20)

        # 3. Grid for Diagnosis Cards
        grid_frame = ctk.CTkFrame(report_win, fg_color="transparent")
        grid_frame.pack(fill="both", expand=True, padx=20, pady=10)
        grid_frame.grid_columnconfigure(0, weight=1)
        grid_frame.grid_columnconfigure(1, weight=1)

        # Card 1: Primary Prediction
        card1 = ctk.CTkFrame(grid_frame, fg_color="#1e2a38", corner_radius=15)
        card1.grid(row=0, column=0, padx=10, sticky="nsew")
        ctk.CTkLabel(card1, text="PRIMARY MATCH", font=("Segoe UI", 12, "bold"), text_color="#5ea8ff").pack(pady=(20, 5))
        ctk.CTkLabel(card1, text=predictions[0][0], font=("Segoe UI", 22, "bold")).pack(pady=10)
        
        conf1_color = "#32a852" if predictions[0][1] > 70 else "#a88e32"
        ctk.CTkLabel(card1, text=f"{predictions[0][1]:.1f}% Confidence", font=("Segoe UI", 18), text_color=conf1_color).pack()

        # Card 2: Secondary Prediction
        card2 = ctk.CTkFrame(grid_frame, fg_color="#2a1e20", corner_radius=15)
        card2.grid(row=0, column=1, padx=10, sticky="nsew")
        ctk.CTkLabel(card2, text="SECONDARY MATCH", font=("Segoe UI", 12, "bold"), text_color="#ff7575").pack(pady=(20, 5))
        ctk.CTkLabel(card2, text=predictions[1][0], font=("Segoe UI", 22, "bold")).pack(pady=10)
        ctk.CTkLabel(card2, text=f"{predictions[1][1]:.1f}% Confidence", font=("Segoe UI", 18), text_color="gray").pack()

        # Footer / Disclaimer
        ctk.CTkLabel(report_win, text="DISCLAIMER: This system utilizes machine learning for educational pre-consultation.\nIt does not replace professional medical advice. Please consult a physician.", 
                     font=("Segoe UI", 11), text_color="#888888", justify="center").pack(side="bottom", pady=20)

        # 4. Finalize session
        self.user_symptoms = [] 
        self.add_log("Assistant", "I have generated your diagnosis report. Your symptom session has been cleared.")

In [139]:
# Update your main block to include the test:
if __name__ == "__main__":
    try:
        engine = MedicalEngine("training.csv")
        
        # Run the test before launching the UI
        engine.run_system_test( "testing.csv")
        
        app = MedicalUI(engine)
        app.mainloop()
    except Exception as e:
        print(f"Error: {e}")


--- INITIATING SYSTEM VALIDATION (testing.csv) ---
STATUS: Validation Complete. Accuracy: 97.62%


KeyboardInterrupt: 